In [ ]:

# =====================================================
# CUSTOMER ETL PIPELINE - CONFIGURATION
# =====================================================

# Database configuration
DATABASE = "API_DATA_ENGINEERING"
WAREHOUSE = "API_PROJECT_WH"

# Pipeline parameters
BATCH_ID = "BATCH_001"
SOURCE_URL = "https://jsonplaceholder.typicode.com/users"

# Execution parameters
LOAD_MODE = "FULL"
RUN_DATE = "2026-11-18"

# Display configuration
print("CUSTOMER ETL PIPELINE CONFIGURATION")
print("-----------------------------------")
print(f"Database     : {DATABASE}")
print(f"Warehouse    : {WAREHOUSE}")
print(f"Batch ID     : {BATCH_ID}")
print(f"Source URL   : {SOURCE_URL}")
print(f"Load Mode    : {LOAD_MODE}")
print(f"Run Date     : {RUN_DATE}")


In [ ]:

# =====================================================
# BRONZE LAYER - SAMPLE API DATA INGESTION
# =====================================================

import json
from datetime import datetime
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, parse_json

# Get Snowflake session
session = get_active_session()

# Simulated API response
customers = [
    {
        "id": 1,
        "name": "John Smith",
        "email": "john@gmail.com",
        "status": "active",
        "address": {"city": "Amsterdam", "zipcode": "1000AA"},
        "company": {"name": "ABC Technologies"}
    },
    {
        "id": 2,
        "name": "Sarah Johnson",
        "email": "sarah@gmail.com",
        "status": "active",
        "address": {"city": "Rotterdam", "zipcode": "3000AA"},
        "company": {"name": "Data Solutions"}
    },
    {
        "id": 3,
        "name": "Michael Brown",
        "email": None,
        "status": "inactive",
        "address": {"city": "Utrecht", "zipcode": "3500AA"},
        "company": {"name": "Cloud Systems"}
    },
    {
        "id": 4,
        "name": "Emma Wilson",
        "email": "emma@gmail.com",
        "status": "active",
        "address": {"city": "Amsterdam", "zipcode": "1000BB"},
        "company": {"name": "ABC Technologies"}
    },
    {
        "id": 5,
        "name": "David Miller",
        "email": "david@gmail.com",
        "status": "inactive",
        "address": {"city": "Eindhoven", "zipcode": "5600AA"},
        "company": {"name": "Innovation Labs"}
    }
]

print(f"Records extracted: {len(customers)}")

# Convert JSON objects into strings
bronze_data = [
    (
        BATCH_ID,
        "SAMPLE_CUSTOMER_API",
        json.dumps(customer),
        datetime.utcnow()
    )
    for customer in customers
]

# Create Snowpark DataFrame
bronze_df = session.create_dataframe(
    bronze_data,
    schema=[
        "BATCH_ID",
        "SOURCE_NAME",
        "RAW_JSON",
        "INGESTED_AT"
    ]
)

# Convert JSON strings into Snowflake VARIANT
bronze_df = bronze_df.select(
    col("BATCH_ID"),
    col("SOURCE_NAME"),
    parse_json(col("RAW_JSON")).alias("RAW_PAYLOAD"),
    col("INGESTED_AT")
)

# Load into Bronze
bronze_df.write.mode("append").save_as_table(
    f"{DATABASE}.BRONZE.RAW_API_CUSTOMERS"
)

print("Bronze loading completed successfully")
print(f"Records loaded: {len(customers)}")


In [ ]:
# =====================================================
# SILVER LAYER - DATA CLEANING AND TRANSFORMATION
# =====================================================

from snowflake.snowpark.functions import (
    col, upper, lower, trim, when, current_timestamp, lit
)

# Read Bronze data for current batch
raw_df = session.table(
    f"{DATABASE}.BRONZE.RAW_API_CUSTOMERS"
).filter(
    col("BATCH_ID") == BATCH_ID
)

# Extract and standardise fields from JSON
silver_df = raw_df.select(
    col("RAW_PAYLOAD")["id"].cast("STRING").alias("CUSTOMER_ID"),
    trim(col("RAW_PAYLOAD")["name"].cast("STRING")).alias("CUSTOMER_NAME"),
    lower(trim(col("RAW_PAYLOAD")["email"].cast("STRING"))).alias("EMAIL"),

    # Country is not available in our sample API data
    lit(None).cast("STRING").alias("COUNTRY"),

    upper(trim(col("RAW_PAYLOAD")["status"].cast("STRING"))).alias("STATUS"),
    trim(col("RAW_PAYLOAD")["address"]["city"].cast("STRING")).alias("CITY"),
    trim(col("RAW_PAYLOAD")["address"]["zipcode"].cast("STRING")).alias("POSTAL_CODE"),
    col("BATCH_ID")
)

# Identify invalid records
invalid_condition = (
    col("CUSTOMER_ID").is_null() |
    col("CUSTOMER_NAME").is_null() |
    (col("CUSTOMER_NAME") == "") |
    col("EMAIL").is_null() |
    (col("EMAIL") == "")
)

# Create rejected records directly from Bronze
reject_df = raw_df.filter(
    col("RAW_PAYLOAD")["email"].cast("STRING").is_null() |
    (trim(col("RAW_PAYLOAD")["email"].cast("STRING")) == "")
)

reject_output = reject_df.select(
    col("BATCH_ID"),
    col("RAW_PAYLOAD"),
    lit("Missing email").alias("REJECTION_REASON"),
    current_timestamp().alias("REJECTED_AT")
)

# Save rejected records
reject_output.write.mode("append").save_as_table(
    f"{DATABASE}.SILVER.CUSTOMER_REJECTS"
)

# Keep valid records
valid_df = silver_df.filter(~invalid_condition)

# Remove duplicate customer IDs within this batch
valid_df = valid_df.drop_duplicates(["CUSTOMER_ID"])

# Add processing timestamp
valid_df = valid_df.with_column(
    "PROCESSED_AT", current_timestamp()
)

# Explicitly match target table column order
valid_df = valid_df.select(
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "EMAIL",
    "COUNTRY",
    "STATUS",
    "CITY",
    "POSTAL_CODE",
    "BATCH_ID",
    "PROCESSED_AT"
)

# Save valid records
valid_df.write.mode("append").save_as_table(
    f"{DATABASE}.SILVER.CUSTOMERS",
    column_order="name"
)

# Execution summary
print("Silver transformation completed successfully")
print(f"Valid records: {valid_df.count()}")
print(f"Rejected records: {reject_output.count()}")

In [ ]:

# =====================================================
# GOLD LAYER - CUSTOMER AGGREGATION
# =====================================================

from snowflake.snowpark.functions import (
    col, when, count, sum as sum_, current_timestamp, coalesce, lit
)

# Read Silver customer data for current batch
silver_df = session.table(
    f"{DATABASE}.SILVER.CUSTOMERS"
).filter(
    col("BATCH_ID") == BATCH_ID
)

# Replace missing country with UNKNOWN
silver_df = silver_df.with_column(
    "COUNTRY",
    coalesce(col("COUNTRY"), lit("UNKNOWN"))
)

# Aggregate customer metrics by country
gold_df = silver_df.group_by("COUNTRY").agg(
    count("*").alias("TOTAL_CUSTOMERS"),
    sum_(
        when(col("STATUS") == "ACTIVE", 1).otherwise(0)
    ).alias("ACTIVE_CUSTOMERS"),
    sum_(
        when(col("STATUS") == "INACTIVE", 1).otherwise(0)
    ).alias("INACTIVE_CUSTOMERS")
)

# Add refresh timestamp
gold_df = gold_df.with_column(
    "REFRESHED_AT", current_timestamp()
)

# Match target table column order
gold_df = gold_df.select(
    "COUNTRY",
    "TOTAL_CUSTOMERS",
    "ACTIVE_CUSTOMERS",
    "INACTIVE_CUSTOMERS",
    "REFRESHED_AT"
)

# Save Gold results
gold_df.write.mode("append").save_as_table(
    f"{DATABASE}.GOLD.CUSTOMER_OVERVIEW",
    column_order="name"
)

print("Gold transformation completed successfully")
print(f"Countries processed: {gold_df.count()}")
